<a href="https://www.kaggle.com/code/niladriroy0/modular-transformer-research-framework?scriptVersionId=308135011" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ======================================================
# RESEARCH-GRADE TRANSFORMER FRAMEWORK (FULL LOGGING)
# Optimized for Kaggle with Causal Masking & Memory Efficiency
# ======================================================

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import time
import math
from itertools import product
from tqdm.auto import tqdm

print("\n================ INITIALIZING FRAMEWORK ================")

# ======================================================
# 1. REPRODUCIBILITY
# ======================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ======================================================
# 2. CONFIG
# ======================================================

class Config:
    def __init__(self, **kwargs):
        self.d_model = 128
        self.num_heads = 8
        self.num_layers = 2
        self.seq_len = 128
        self.batch_size = 64
        self.lr = 3e-4
        self.epochs = 10
        self.dropout = 0.1
        self.weight_decay = 0.01
        self.grad_clip = 1.0
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Grid parameters
        self.attention_type = "standard"
        self.pos_encoding = "learned"
        self.activation = "relu"
        self.ff_multiplier = 4

        for k, v in kwargs.items():
            setattr(self, k, v)

# ======================================================
# 3. TOKENIZER & DATASET (Slicing Optimized)
# ======================================================

class CharTokenizer:
    def __init__(self, text):
        print("[Tokenizer] Building vocabulary...")
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        print(f"[Tokenizer] Vocabulary size: {self.vocab_size}")
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, text):
        return [self.stoi[ch] for ch in text if ch in self.stoi]


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, text, tokenizer, seq_len):
        encoded = tokenizer.encode(text)
        self.data = torch.tensor(encoded, dtype=torch.long)
        self.seq_len = seq_len
        self.num_samples = (len(self.data) - 1) // seq_len
        print(f"[Dataset] Total samples available: {self.num_samples}")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        x = self.data[start:end]
        y = self.data[start+1:end+1]
        return x, y

# ======================================================
# 4. MODEL (Causal Masking & Logic Switch)
# ======================================================

class TransformerModel(nn.Module):
    def __init__(self, config, vocab_size):
        super().__init__()
        self.config = config
        
        self.embedding = nn.Embedding(vocab_size, config.d_model)
        
        if config.pos_encoding == "learned":
            self.pos_embedding = nn.Embedding(config.seq_len, config.d_model)
        else:
            self.pos_embedding = nn.Parameter(torch.zeros(1, config.seq_len, config.d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.num_heads,
            dim_feedforward=config.d_model * config.ff_multiplier,
            dropout=config.dropout,
            activation=config.activation,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, config.num_layers)

        self.norm = nn.LayerNorm(config.d_model)
        self.output = nn.Linear(config.d_model, vocab_size)

    def forward(self, x):
        sz = x.size(1)
        mask = torch.triu(torch.ones(sz, sz, device=x.device) * float('-inf'), diagonal=1)
        positions = torch.arange(0, sz, device=x.device).unsqueeze(0)
        x = (self.embedding(x) * math.sqrt(self.config.d_model)) + self.pos_embedding(positions)
        x = self.transformer(x, mask=mask)
        x = self.norm(x)
        return self.output(x)

# ======================================================
# 5. TRAINING + FULL LOGGING
# ======================================================

def evaluate(model, loader, config):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(config.device), y.to(config.device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0
    return avg_loss, math.exp(min(avg_loss, 20))


def train_model(model, train_loader, val_loader, config, exp_id, seed):
    model.to(config.device)
    optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    criterion = nn.CrossEntropyLoss()

    epoch_logs = []
    batch_logs = []
    start_time = time.time()

    for epoch in range(config.epochs):
        model.train()
        running_loss = 0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Exp {exp_id} Seed {seed} Ep {epoch+1}", leave=False)
        
        for batch_idx, (x, y) in pbar:
            x, y = x.to(config.device), y.to(config.device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            running_loss += loss.item()

            batch_logs.append({
                "level": "batch", "experiment": exp_id, "seed": seed,
                "epoch": epoch+1, "batch": batch_idx, "train_loss": loss.item(),
                "val_loss": None, "val_perplexity": None,
                "learning_rate": optimizer.param_groups[0]['lr'], "train_time": None,
                "activation": config.activation,
                "dropout": config.dropout,
                "lr_config": config.lr,
                "ff_multiplier": config.ff_multiplier
            })
            
            if batch_idx % 10 == 0:
                pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()
        train_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0
        val_loss, val_ppl = evaluate(model, val_loader, config)

        epoch_logs.append({
            "level": "epoch", "experiment": exp_id, "seed": seed,
            "epoch": epoch+1, "batch": None, "train_loss": train_loss,
            "val_loss": val_loss, "val_perplexity": val_ppl,
            "learning_rate": optimizer.param_groups[0]['lr'], "train_time": None,
            "activation": config.activation,
            "dropout": config.dropout,
            "lr_config": config.lr,
            "ff_multiplier": config.ff_multiplier
        })
        print(f"[Experiment {exp_id}, Seed {seed}, Epoch {epoch+1}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | PPL: {val_ppl:.2f} 🤯")

    print("-------------------------------------------------------------------------------")
    total_time = time.time() - start_time
    for row in epoch_logs:
        row["train_time"] = total_time
    return epoch_logs, batch_logs

# ======================================================
# 6. DATA LOADING
# ======================================================

print("\n================ LOADING DATA ================")
DATA_PATH = "/kaggle/input/datasets/thedevastator/the-bards-best-a-character-modeling-dataset"

try:
    train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
    val_df = pd.read_csv(f"{DATA_PATH}/validation.csv")
    train_text = "\n".join(train_df["text"].astype(str).tolist())
    val_text = "\n".join(val_df["text"].astype(str).tolist())
    print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)}")
except Exception as e:
    print(f"Data loading error: {e}. Using dummy data.")
    train_text = "to be or not to be that is the question " * 1000
    val_text = "shall i compare thee to a summers day " * 200

tokenizer = CharTokenizer(train_text)
train_dataset = TextDataset(train_text, tokenizer, seq_len=128)
val_dataset = TextDataset(val_text, tokenizer, seq_len=128)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, drop_last=True)

# ======================================================
# 7. RUN EXPERIMENTS
# ======================================================

SEEDS = [42, 999]

grid_params = {
    "attention_type": ["standard"],
    "pos_encoding": ["learned"],
    # --- DEPTH SCALING ---
    "num_layers": [4, 6],

    # --- WIDTH SCALING ---
    "d_model": [128, 256, 512],

    # --- HEAD VARIATION (Keep divisible with d_model) ---
    "num_heads": [4, 8],

    # --- ACTIVATION ---
    "activation": ["relu", "gelu"],

    # --- REGULARIZATION ---
    "dropout": [0.1],

    # --- LEARNING RATE STABILITY ---
    "lr": [3e-4],

    # --- FF CAPACITY ---
    "ff_multiplier": [8]
}

keys, values = zip(*grid_params.items())
experiment_grid = [dict(zip(keys, v)) for v in product(*values)]

print(f"[Grid] Total unique configurations: {len(experiment_grid)}")
print(f"[Grid] Total runs (Configs x Seeds): {len(experiment_grid) * len(SEEDS)}")

MASTER_LOGS = []

for exp_id, exp in enumerate(experiment_grid, 1):
    print(f"\n>>>> Starting Experiment {exp_id}/{len(experiment_grid)} | Config: {exp}")
    for seed in SEEDS:
        set_seed(seed)
        config = Config(**exp)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = TransformerModel(config, vocab_size=tokenizer.vocab_size)
        epoch_logs, batch_logs = train_model(model, train_loader, val_loader, config, exp_id, seed)
        MASTER_LOGS.extend(epoch_logs)
        MASTER_LOGS.extend(batch_logs)

print("\n================ ALL EXPERIMENTS COMPLETED ================")
FINAL_DF = pd.DataFrame(MASTER_LOGS)
FINAL_DF.to_csv("full_experiment_log.csv", index=False)

print(f"\n===== LOGGING COMPLETE =====")
print(FINAL_DF.groupby(['experiment', 'seed']).tail(1)[['experiment', 'seed', 'val_loss', 'val_perplexity']])

# ======================================================
# 8. RESEARCH-STYLE SUMMARY (Mean ± Std Across Seeds)
# ======================================================

summary = FINAL_DF[FINAL_DF["level"] == "epoch"].groupby(["experiment"]).agg({"val_loss": ["mean", "std"],"val_perplexity": ["mean", "std"]})

print("\n===== MEAN ± STD SUMMARY ACROSS SEEDS =====")
print(summary)



================ INITIALIZING FRAMEWORK ================

================ LOADING DATA ================
Train rows: 1 | Val rows: 1
[Tokenizer] Building vocabulary...
[Tokenizer] Vocabulary size: 65
[Dataset] Total samples available: 7842
[Dataset] Total samples available: 435
[Grid] Total unique configurations: 24
[Grid] Total runs (Configs x Seeds): 48

>>>> Starting Experiment 1/24 | Config: {'attention_type': 'standard', 'pos_encoding': 'learned', 'num_layers': 4, 'd_model': 128, 'num_heads': 4, 'activation': 'relu', 'dropout': 0.1, 'lr': 0.0003, 'ff_multiplier': 8}


Exp 1 Seed 42 Ep 1:   0%|          | 0/122 [00:00<?, ?it/s]

[Experiment 1, Seed 42, Epoch 1] Train Loss: 2.7390 | Val Loss: 2.5014 | PPL: 12.20 🤯


Exp 1 Seed 42 Ep 2:   0%|          | 0/122 [00:00<?, ?it/s]

[Experiment 1, Seed 42, Epoch 2] Train Loss: 2.4708 | Val Loss: 2.4377 | PPL: 11.45 🤯


...

[Experiment 24, Seed 999, Epoch 8] Train Loss: 1.5179 | Val Loss: 1.5993 | PPL: 4.95 🤯


Exp 24 Seed 999 Ep 9:   0%|          | 0/122 [00:00<?, ?it/s]

[Experiment 24, Seed 999, Epoch 9] Train Loss: 1.4949 | Val Loss: 1.5838 | PPL: 4.87 🤯


Exp 24 Seed 999 Ep 10:   0%|          | 0/122 [00:00<?, ?it/s]

[Experiment 24, Seed 999, Epoch 10] Train Loss: 1.4822 | Val Loss: 1.5785 | PPL: 4.85 🤯
-------------------------------------------------------------------------------

================ ALL EXPERIMENTS COMPLETED ================

===== LOGGING COMPLETE =====
       experiment  seed  val_loss  val_perplexity
1229            1    42       NaN             NaN
2459            1   999       NaN             NaN
3689            2    42       NaN             NaN
4919            2   999       NaN             NaN
6149            3    42       NaN             NaN
7379            3   999       NaN             NaN
8609            4    42       NaN             NaN
9839            4   999       NaN             NaN
11069           5    42       NaN             NaN
12299           5   999       NaN             NaN
13529           6    42       NaN             NaN
14759           6   999       NaN             NaN
15989           7    42       NaN             NaN
17219           7   999       NaN        

In [2]:
FINAL_DF.head(50)

,level,experiment,seed,epoch,batch,train_loss,val_loss,val_perplexity,learning_rate,train_time,activation,dropout,lr_config,ff_multiplier
0,epoch,1,42,1,NaN,2.738967,2.501355,12.199011,0.000293,34.955217,relu,0.1,0.0003,8
1,epoch,1,42,2,NaN,2.470787,2.437711,11.446805,0.000271,34.955217,relu,0.1,0.0003,8
2,epoch,1,42,3,NaN,2.415581,2.398356,11.005064,0.000238,34.955217,relu,0.1,0.0003,8
3,epoch,1,42,4,NaN,2.374777,2.347683,10.461299,0.000196,34.955217,relu,0.1,0.0003,8
4,epoch,1,42,5,NaN,2.332568,2.291957,9.894280,0.000150,34.955217,relu,0.1,0.0003,8
5,epoch,1,42,6,NaN,2.284519,2.240416,9.397244,0.000104,34.955217,relu,0.1,0.0003,8
6,epoch,1,42,7,NaN,2.245776,2.205997,9.079295,0.000062,34.955217,relu,0.1,0.0003,8
7,epoch,1,42,8,NaN,2.220053,2.184572,8.886846,0.000029,34.955217,relu,0.1,0.0003,8
8,epoch,1,42,9,NaN,2.205682,2.174990,8.802100,0.000007,34.955217,relu,0.1,0.0003,8
9,epoch,1,42,10,NaN,2.199656,2.172010,8.775908,0.000000,34.955217,relu,0.1,0.0003,8


In [3]:
FINAL_DF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59040 entries, 0 to 59039
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   level           59040 non-null  object 
 1   experiment      59040 non-null  int64  
 2   seed            59040 non-null  int64  
 3   epoch           59040 non-null  int64  
 4   batch           58560 non-null  float64
 5   train_loss      59040 non-null  float64
 6   val_loss        480 non-null    float64
 7   val_perplexity  480 non-null    float64
 8   learning_rate   59040 non-null  float64
 9   train_time      480 non-null    float64
 10  activation      59040 non-null  object 
 11  dropout         59040 non-null  float64
 12  lr_config       59040 non-null  float64
 13  ff_multiplier   59040 non-null  int64  
dtypes: float64(8), int64(4), object(2)
memory usage: 6.3+ MB
